# The expression tree and its interpreters

A model in axiom is a tree of typed `Spec` nodes. A regression is `Add(Mul(Param, Data), ...)`,
a structural model is a `System`, carryover is a `Convolve`, a differential equation is an
`ODESystem`. One tree, several interpreters:

| Interpreter | Produces |
|---|---|
| `dimension` | a `Dimension`, or a `DimensionError` naming the node |
| `value` | a numpy array — this *is* `forward()` |
| `latex` | the rendered equation |

The dimension checker is abstract interpretation over the **same tree** the likelihood
evaluates, so there is no separate declaration of a model's units that can drift from the
model. That is rule 3 ("one `forward()`") generalized from one function to one representation.

In [ ]:
from fractions import Fraction

import numpy as np

from axiom.core import (
    Add, Apply, ApplyFn, Const, Convolve, D, Data, DimensionError, Div, Equation, Expr, Link, LinkFn,
    Model, Mul, ODESystem, Opaque, OpaqueFn, OpaqueRegistry, Param, Pow, Prior, Spec, SupportsForward,
    System, causal_convolve, check, children, data_names, dimension, dimensionless, latex,
    latex_or_unsupported, node_path, params, value, walk,
)

## Leaves: `Const`, `Data`, `Param`

Every leaf declares its dimension. A `Param` with a dimensionless dimension is a **shape**
parameter (it pools across studies); one with a dose or time dimension is a **scale**.

In [ ]:
dose = Data(name="dose", dimension=D.currency)
k = Param(name="k", dimension=D.currency, prior=Prior(family="lognormal", hyper={"mu": 3.0, "sigma": 1.0}))
s = Param(name="s", dimension=dimensionless(), prior=Prior(family="gamma", hyper={"alpha": 2.0, "beta": 1.0}))
beta = Param(name="beta", dimension=D.outcome)
one = Const(value=1.0, dimension=dimensionless())

print("k is a scale:", not k.is_shape, "| s is a shape:", s.is_shape)

## A Hill curve

`Pow` with a *variable* exponent needs a dimensionless base — so the dose must be divided by
the half-saturation scale `k` first. That forced `x / k` is the whole scale/shape story.

In [ ]:
x = Div(numerator=dose, denominator=k)
hill: Expr = Mul(factors=(beta, Div(numerator=Pow(base=x, exponent=s),
                                    denominator=Add(terms=(one, Pow(base=x, exponent=s))))))

print(dimension(hill))
print(latex(hill))

In [ ]:
d = np.array([0.0, 25.0, 50.0, 100.0, 200.0])
value(hill, data={"dose": d}, params={"k": 50.0, "s": 2.0, "beta": 10.0})

Parameters can be posterior draws — arrays broadcast by numpy's rules, so one call evaluates
the curve under every draw.

In [ ]:
rng = np.random.default_rng(0)
theta = {"k": rng.lognormal(np.log(50), 0.2, size=(500, 1)), "s": rng.gamma(8, 0.25, size=(500, 1)), "beta": 10.0}
curves = value(hill, data={"dose": d[None, :]}, params=theta)
print(curves.shape, "| mean at each dose:", curves.mean(axis=0).round(2))

## The checker names the node

A `log` of a dimensioned quantity, a sum of unlike dimensions, a dimensioned convolution
kernel — each fails at construction-time checking with the path to the offending node.

In [ ]:
bad_trees: list[Model] = [
    Apply(fn="log", arg=dose),
    Add(terms=(beta, Mul(factors=(beta, Apply(fn="log", arg=dose))))),
    Pow(base=dose, exponent=s),
    Convolve(signal=dose, kernel=k),
]
for t in bad_trees:
    try:
        dimension(t)
    except DimensionError as e:
        print(f"{type(t).__name__:9s} -> {e}")

A rational *constant* exponent is fine on any base (review A3): the square root of a variance
is a non-integer power of a dimensioned quantity.

In [ ]:
print(dimension(Pow(base=Mul(factors=(dose, dose)), exponent=Fraction(1, 2))))
print(dimension(Pow(base=dose, exponent="1/3")), "|", latex(Pow(base=dose, exponent=0.5)))

## `check` against a declaration

`check(model, expected)` is what an `Estimand` or a kernel calls to assert the tree derives
to what it claims.

In [ ]:
print(check(hill, D.outcome))
try:
    check(hill, D.outcome / D.currency)
except DimensionError as e:
    print("refused:", e)

## Transcendentals and links

`Apply` takes an `ApplyFn`; `Link` takes a `LinkFn`. Both require a dimensionless argument
and return a dimensionless value — which is why a log-outcome model needs a reference divide
(`log(y / y_ref)`), and why D9 in the plan exists.

In [ ]:
fns: list[ApplyFn] = ["exp", "sigmoid", "softplus"]
for fn in fns:
    print(fn, value(Apply(fn=fn, arg=Const(value=0.0, dimension=dimensionless()))))

link: LinkFn = "log"
y = Data(name="y", dimension=D.outcome)
y_ref = Const(value=100.0, dimension=D.outcome)
print(dimension(Link(fn=link, arg=Div(numerator=y, denominator=y_ref))))

## Carryover as `Convolve`, with an `Opaque` kernel

`Convolve` applies dimensionless weights causally along the last axis. Here the weights come
from an `Opaque` node — user code the tree cannot express. It declares its dimension and is
evaluated through an `OpaqueRegistry`; anything needing introspection (LaTeX) degrades to a
typed `Unsupported`.

In [ ]:
lam = Param(name="lam", dimension=dimensionless())
weights = Opaque(name="geometric", inputs=(lam,), dimension=dimensionless())
carry = Convolve(signal=dose, kernel=weights)

geometric: OpaqueFn = lambda lam, L=6: lam ** np.arange(L)
registry: OpaqueRegistry = {"geometric": geometric}

print(dimension(carry))
print(value(carry, data={"dose": np.array([100.0, 0, 0, 0, 0, 0])}, params={"lam": 0.6}, opaque=registry))
print(latex_or_unsupported(carry))
print(causal_convolve(np.ones(4), np.array([0.5, 0.25])))

## Equations, systems, ODEs

`Equation` requires both sides to share a dimension. `System` checks each equation.
`ODESystem` checks `dim(rhs_i) == dim(state_i) / dim(t)` — the per-period-versus-cumulative
bug class, caught before anything is sampled. In 1.0 the ODE node is dimension-check-only
(review C1).

In [ ]:
response = Equation(lhs=y, rhs=Add(terms=(Param(name="a", dimension=D.outcome), hill)), name="response")
cost = Equation(lhs=Data(name="cost", dimension=D.currency), rhs=Mul(factors=(dose, Const(value=1.0, dimension=dimensionless()))))
model = System(equations=(response, cost))
print(dimension(model))
print(latex(model))
print(value(model, data={"dose": d}, params={"k": 50.0, "s": 2.0, "beta": 10.0, "a": 1.0}).shape)

In [ ]:
S = Data(name="S", dimension=D.outcome)
t = Data(name="t", dimension=D.time)
r = Param(name="r", dimension=D.time ** -1)
ode = ODESystem(states=(S,), rhs=(Mul(factors=(r, S)),), time=t)
print(dimension(ode), "|", latex(ode))

try:
    dimension(ODESystem(states=(S,), rhs=(S,), time=t))   # forgot the rate: dS/dt = S
except DimensionError as e:
    print("refused:", e)

## Traversal and serialization

Helpers for walking a tree; and because every node is a `Spec`, a whole model round-trips as
JSON and has a content hash.

In [ ]:
print([p.name for p in params(model)], data_names(model))
print(children(hill)[0], "|", node_path(hill, s))
print([path for path, _ in walk(hill)][:5])
print(Spec.from_json(model.to_json()) == model, model.content_hash()[:16])

## `SupportsForward`

The design layer depends on this protocol: a thing with an `expr` and a `forward` that is the
value interpreter over it. Implementations are not allowed to re-implement the transform chain.

In [ ]:
class HillSurface:
    expr = hill

    def forward(self, dose, theta):
        return value(self.expr, data=dose, params=theta)

surface = HillSurface()
print(isinstance(surface, SupportsForward), surface.forward({"dose": d}, {"k": 50.0, "s": 2.0, "beta": 10.0}))